# LogisChain AI — 03: Model Training

Train XGBoost, LightGBM, Survival, and Ensemble models.

**Goals:**
- Train XGBoost and LightGBM on carrier default prediction
- SHAP feature importance analysis
- ROC Curve visualization
- Ensemble stacking

In [ ]:
!rm -rf /content/logischain-ai-8
!git clone https://github.com/ZethetaIntern/logischain-ai-8.git -q
!pip install mlflow lightgbm shap optuna tqdm faker lifelines -q
import sys
sys.path.insert(0, '/content/logischain-ai-8')
print('Setup done!')

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
from src.data.pipeline import SyntheticDataGenerator
from src.features.fusion_features import FeaturePipeline
from src.utils.metrics import classification_report_dict
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
import lightgbm as lgb

mlflow.set_experiment('logischain_ai')
gen = SyntheticDataGenerator(seed=42)
data = gen.generate_all()
print('Data ready.')

In [ ]:
fp = FeaturePipeline()
fused = fp.run(data['carriers'], data['shipments'], data['financial'])
target_col = 'carrier_failure'
y = fused[target_col].fillna(0)
drop_cols = [target_col, 'carrier_id', 'company_id', 'carrier_type', 'region', 'industry']
X = fused.drop(columns=[c for c in drop_cols if c in fused.columns]).select_dtypes(include=np.number).fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
with mlflow.start_run(run_name='xgboost_carrier_default'):
    xgb_model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
    xgb_model.fit(X_train, y_train)
    xgb_probs = xgb_model.predict_proba(X_test)[:,1]
    xgb_metrics = classification_report_dict(y_test.values, xgb_probs)
    mlflow.log_metrics({k: v for k, v in xgb_metrics.items() if isinstance(v, float)})
    print('XGBoost:', {k: round(v, 4) for k, v in xgb_metrics.items() if k in ['roc_auc', 'f1', 'ks_statistic']})

In [ ]:
with mlflow.start_run(run_name='lightgbm_carrier_default'):
    lgb_model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
    lgb_model.fit(X_train, y_train)
    lgb_probs = lgb_model.predict_proba(X_test)[:,1]
    lgb_metrics = classification_report_dict(y_test.values, lgb_probs)
    mlflow.log_metrics({k: v for k, v in lgb_metrics.items() if isinstance(v, float)})
    print('LightGBM:', {k: round(v, 4) for k, v in lgb_metrics.items() if k in ['roc_auc', 'f1', 'ks_statistic']})

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
S_train = np.column_stack([xgb_model.predict_proba(X_train)[:,1], lgb_model.predict_proba(X_train)[:,1]])
S_test = np.column_stack([xgb_probs, lgb_probs])
meta = LogisticRegression(random_state=42)
meta.fit(S_train, y_train)
ens_probs = meta.predict_proba(S_test)[:,1]
ens_auc = roc_auc_score(y_test, ens_probs)
print(f'Ensemble AUC: {ens_auc:.4f}')

In [ ]:
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ROC Curves
for name, probs in [('XGBoost', xgb_probs), ('LightGBM', lgb_probs), ('Ensemble', ens_probs)]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, lw=2, label=f'{name} (AUC={roc_auc:.3f})')
axes[0].plot([0,1],[0,1],'k--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves - Model Comparison')
axes[0].legend()

# Feature Importance
importances = xgb_model.feature_importances_
top_idx = np.argsort(importances)[-10:]
axes[1].barh(range(10), importances[top_idx], color='steelblue')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([X_train.columns[i] for i in top_idx], fontsize=8)
axes[1].set_title('Top 10 Feature Importances (XGBoost)')
axes[1].set_xlabel('Importance')

# Model Comparison Bar Chart
models = ['XGBoost', 'LightGBM', 'Ensemble']
aucs = [roc_auc_score(y_test, xgb_probs), roc_auc_score(y_test, lgb_probs), ens_auc]
bars = axes[2].bar(models, aucs, color=['steelblue', 'orange', 'green'])
axes[2].set_ylim(0.7, 0.95)
axes[2].set_title('Model AUC Comparison')
axes[2].set_ylabel('AUC-ROC')
for bar, val in zip(bars, aucs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{val:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('Model training and visualization complete!')